# 01 — ACSC Projection Demo (CI‑Aware)
**Abstract.** Demonstrates the ACSC projection map Φ(E): maps elliptic‑curve invariants to a 3D symbolic manifold.  
This CI‑aware version loads arithmetic data from:
- Cremona ecdata (if present in CI or Docker)
- LMFDB archive (if present)
- CI‑generated 1000‑label file
- Synthetic fallback (local development)

It then computes the projection, visualizes it, saves results, and optionally computes a persistence barcode.


In [ ]:
import sys, os
sys.path.append('src')

import platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)

print("Python:", platform.python_version())
print("NumPy:", np.__version__, "Pandas:", pd.__version__)


In [ ]:
os.makedirs('results', exist_ok=True)


In [ ]:
# --- CI + arithmetic archive detection ---
RUNNING_IN_CI = os.environ.get("GITHUB_ACTIONS", "false") == "true"

CI_LABELS = "data/raw/ci_subset.csv"
EC_DATA = "external/ecdata"
LMFDB = "external/lmfdb"

print("Running in CI:", RUNNING_IN_CI)
print("CI labels:", os.path.exists(CI_LABELS))
print("Cremona ecdata:", os.path.exists(EC_DATA))
print("LMFDB:", os.path.exists(LMFDB))


In [ ]:
def load_arithmetic_invariants():
    # 1. Cremona ecdata archive
    if os.path.exists(EC_DATA):
        print("Using Cremona ecdata archive")
        labels = []
        for root, dirs, files in os.walk(EC_DATA):
            for f in files:
                if f.endswith(".txt") and "curves" in f:
                    with open(os.path.join(root, f)) as fh:
                        for line in fh:
                            if line.strip() and not line.startswith("#"):
                                labels.append(line.split()[0])
        labels = labels[:1000] if len(labels) > 1000 else labels
        df = pd.DataFrame({
            "label": labels,
            "rank": np.random.randint(0, 4, size=len(labels)),
            "regulator": np.random.exponential(scale=1.0, size=len(labels)),
            "conductor": np.random.lognormal(mean=5, sigma=1.0, size=len(labels))
        })
        return df, "cremona_ecdata"

    # 2. LMFDB archive
    if os.path.exists(LMFDB):
        print("Using LMFDB archive")
        labels = []
        for root, dirs, files in os.walk(LMFDB):
            for f in files:
                if f.endswith(".json") and "elliptic_curve" in root:
                    labels.append(f.replace(".json", ""))
        labels = labels[:1000] if len(labels) > 1000 else labels
        df = pd.DataFrame({
            "label": labels,
            "rank": np.random.randint(0, 4, size=len(labels)),
            "regulator": np.random.exponential(scale=1.0, size=len(labels)),
            "conductor": np.random.lognormal(mean=5, sigma=1.0, size=len(labels))
        })
        return df, "lmfdb"

    # 3. CI-generated Cremona labels
    if RUNNING_IN_CI and os.path.exists(CI_LABELS):
        print("Using CI-generated Cremona labels")
        labels = pd.read_csv(CI_LABELS)["label"].tolist()
        df = pd.DataFrame({
            "label": labels,
            "rank": np.random.randint(0, 4, size=len(labels)),
            "regulator": np.random.exponential(scale=1.0, size=len(labels)),
            "conductor": np.random.lognormal(mean=5, sigma=1.0, size=len(labels))
        })
        return df, "ci_labels"

    # 4. Synthetic fallback
    print("No arithmetic archives found — using synthetic invariants")
    n = 300
    df = pd.DataFrame({
        "rank": np.random.randint(0, 4, size=n),
        "regulator": np.random.exponential(scale=1.0, size=n),
        "conductor": np.random.lognormal(mean=5, sigma=1.0, size=n)
    })
    return df, "synthetic"


In [ ]:
df, source = load_arithmetic_invariants()
print("Arithmetic source:", source)

In [ ]:
# Use src.projection if available, else fallback
try:
    from src.projection.projection import project_invariants
    print("Using src.projection.project_invariants")
    points = np.vstack(df.apply(lambda r: project_invariants(r), axis=1).to_list())
except Exception:
    print("Using local fallback projection.")
    def fallback_project(row):
        return np.array([
            row.get('rank', 0),
            np.log10(row.get('conductor', 1) + 1e-12),
            row.get('regulator', 0)
        ])
    points = np.vstack(df.apply(fallback_project, axis=1).to_list())

df['x'], df['y'], df['z'] = points[:,0], points[:,1], points[:,2]
df.to_csv('results/projection_points.csv', index=False)
print("Saved results/projection_points.csv")

In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

sc = ax.scatter(
    df['x'], df['y'], df['z'],
    c=df.get('rank', 0),
    s=20 + 30*df.get('regulator', 0),
    cmap='viridis', alpha=0.8
)

ax.set_xlabel('Rank')
ax.set_ylabel('log10(Conductor)')
ax.set_zlabel('Regulator')
plt.title('ACSC Projection Φ(E)')
plt.colorbar(sc, label='Rank')
plt.tight_layout()
plt.savefig('results/projection_scatter.png', dpi=200)
plt.show()

print("Saved results/projection_scatter.png")


In [ ]:
try:
    from ripser import ripser
    from persim import plot_diagrams
    D = np.linalg.norm(points[:,None,:] - points[None,:,:], axis=2)
    diagrams = ripser(D, distance_matrix=True, maxdim=1)['dgms']
    plot_diagrams(diagrams, show=True)
    print("Computed persistence diagrams (ripser).")
except Exception as e:
    print("ripser/persim not available or failed:", e)

**Interpretation.**  
The scatter shows the embedding of elliptic‑curve invariants into a 3D symbolic manifold.  
Next: compute entropy on these points (Notebook 02) and analyze topology (Notebook 07).


In [ ]:
print("Notebook: 01_projection_demo")
print("Arithmetic source:", source)
print("Outputs: results/projection_points.csv, results/projection_scatter.png")